# 01 · Source mapping + readiness assessment

Map your business's source tables to the canonical roles this asset expects, then assess
readiness (existence, row counts, date span, freshness, empties). Roles that are required
but not yet available in the source (map value `None`) surface as RED, so nothing is
silently missing. Every role is a source input that flows into the twins. Results are
displayed inline so you can see what is ready, what is empty, and what must be ingested or
synthesized before forecasting or building any twin.

This is where most cross-business variation lives. Edit the **Config** cell; the logic is
stable.

## Config — source mapping
`canonical_role -> your source table`. Defaults = CoffCoff (`mickmickstableaws_catalog.gold`).
Set a value to `None`/empty to mark a role as not-yet-available.

In [0]:
dbutils.widgets.text("source_catalog", "mickmickstableaws_catalog", "Source catalog")
dbutils.widgets.text("source_schema", "gold", "Source schema")
dbutils.widgets.text("target_catalog", "mickmickstableaws_catalog", "Target catalog")
dbutils.widgets.text("monitoring_schema", "monitoring", "Monitoring schema")

SRC_CAT = dbutils.widgets.get("source_catalog").strip()
SRC_SCH = dbutils.widgets.get("source_schema").strip()
FQ_MON = f"{dbutils.widgets.get('target_catalog').strip()}.{dbutils.widgets.get('monitoring_schema').strip()}"

def src(table: str) -> str:
    return f"{SRC_CAT}.{SRC_SCH}.{table}"

# canonical role -> source table name (None if not available yet)
SOURCE_MAP = {
    "sales_orders":            "salesorders",
    "demand_plan":             "demandplan",          # forecast target (may be empty)
    "inventory_snapshots":     "inventorysnapshots",
    "inventory_ledger":        "inventoryledger",
    "inventory_opening_stock": "inventoryopeningstock",
    "vendor_performance":      "vendorperformance",
    "production_orders":       "productionorders",
    "production_order_headers":"productionordersheader",
    "production_confirmations":"productionconfirmations",
    "goods_movements":         "goodsmovements",
    "bill_of_materials":       "billofmaterials",
    "work_centres":            "workcentres",
    "outbound_deliveries":     "outbounddeliveries",
    "billing_invoices":        "billinginvoices",
    "materials":               "materialsmd",
    "customers":               "customersmd",
    "product_hierarchies":     "producthierarchiesmd",
    # Required source inputs that are NOT in gold today — all source data flows into the
    # twins, so a missing twin input is a missing source. Leave None until ingested or
    # synthesized for the POC. (Planning-layer inputs like planning_parameters are not
    # tracked here.)
    "routing_master":              None,   # Production twin: operation sequence + standard times
    "setup_matrix":                None,   # Production twin: sequence-dependent changeover matrix
    "distribution_network_master": None,   # Distribution twin: outlet/DC nodes + replenishment edges
    "shipments":                   None,   # Distribution twin: carrier/lane/freight + ship/arrive timestamps
}

# Optional: the date column to use for freshness per role (None = skip freshness).
# TODO — EXTEND for your business: point each role at its primary business-event date column.
FRESHNESS_DATE_COL = {
    "sales_orders":            "CreationDate_ERDAT",
    "vendor_performance":      "PostingDateInTheDocument_BUDAT",
    "production_confirmations": "PostingDate_BUDAT",
    "goods_movements":         "PostingDate_BUDAT",
    "outbound_deliveries":     None,   # set to your delivery date column
    "inventory_ledger":        "PostingDate_BUDAT",
}

## Assess each mapped role
For each role: does the table exist, how many rows, what is the date span, is it empty?

In [0]:
def table_exists(fq: str) -> bool:
    try:
        spark.sql(f"DESCRIBE TABLE {fq}")
        return True
    except Exception:
        return False

rows = []

for role, tbl in SOURCE_MAP.items():
    if not tbl:
        rows.append((role, None, "RED", "CRITICAL",
                     "missing — not available in source (map a table, ingest, or synthesize)",
                     None, None, None))
        continue
    fq = src(tbl)
    if not table_exists(fq):
        rows.append((role, fq, "RED", "CRITICAL", "table not found", None, None, None))
        continue
    cnt = spark.sql(f"SELECT COUNT(*) c FROM {fq}").collect()[0]["c"]
    date_col = FRESHNESS_DATE_COL.get(role)
    min_d = max_d = None
    if date_col:
        try:
            r = spark.sql(f"SELECT MIN(`{date_col}`) mn, MAX(`{date_col}`) mx FROM {fq}").collect()[0]
            min_d, max_d = r["mn"], r["mx"]
        except Exception:
            pass
    if cnt == 0:
        status, sev, msg = "AMBER", "HIGH", "table exists but is EMPTY — must be populated"
    else:
        status, sev, msg = "GREEN", "LOW", f"{cnt} rows"
    rows.append((role, fq, status, sev, msg, cnt, min_d, max_d))

assess = spark.createDataFrame(
    rows,
    schema="role string, table string, status string, severity string, message string, "
           "row_count bigint, min_date date, max_date date",
)
display(assess.orderBy("status", "role"))

## Readiness summary
RED = missing (no table / not available), AMBER = empty (must populate), GREEN = present
with rows. Every role here is a source input that flows into the twins.

Expected for CoffCoff:
- `demand_plan` is **AMBER (empty)** — why `02_generate_demand_forecast` runs next.
- `routing_master`, `setup_matrix` (Production twin) and `distribution_network_master`,
  `shipments` (Distribution twin) are **RED** — required source inputs absent from gold
  today; they must be ingested or synthesized before those twins can be fully built.

In [0]:
display(assess.groupBy("status").count().orderBy("status"))